# Data Exploration

This notebook explores the raw CSV files for the Open University Learning Analytics style dataset and shows how to create a student-level flat table for later analysis.

The Open University context matters because these records describe distance-learning students. VLE clicks, assessment submissions, registration dates, demographics, and final outcomes can be analyzed together to understand student engagement, withdrawal risk, and academic performance in online study.

Main grain for the flat file: one row per `id_student`, `code_module`, and `code_presentation`.

## Data dictionary

Important column meanings:

- `code_module`: module identifier.
- `code_presentation`: presentation identifier, such as `2013B` for February or `2013J` for October.
- `module_presentation_length`: length of the module presentation in days.
- `id_assessment`: assessment identifier.
- `assessment_type`: assessment type: Tutor Marked Assessment (`TMA`), Computer Marked Assessment (`CMA`), or `Exam`.
- `date`: relative day number from the start of the module presentation. In `studentVle`, this is the interaction date; in `assessments`, this is the assessment due date.
- `weight`: assessment weight as a percentage.
- `id_student`: student identifier.
- `date_submitted`: student submission date, relative to the start of the module presentation.
- `is_banked`: whether an assessment result was transferred from a previous presentation.
- `score`: assessment score from 0 to 100; scores below 40 are considered failing scores.
- `gender`, `region`, `highest_education`, `imd_band`, `age_band`, `disability`: student demographic/context fields.
- `num_of_prev_attempts`: previous attempts by the student for the module.
- `studied_credits`: total credits the student was studying at the time.
- `date_registration` and `date_unregistration`: registration/unregistration dates relative to the module start.
- `id_site`: VLE material identifier.
- `sum_click`: number of interactions with a VLE material on a day.
- `activity_type`: VLE material role or category.
- `week_from` and `week_to`: planned usage window for the VLE material.
- `final_result`: final student outcome for the module presentation.

Note: the actual CSV headers are `module_presentation_length` and `assessment_type`.

In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path('../data/raw').resolve()
PROCESSED_DIR = Path('../data/processed').resolve()

csv_files = sorted(RAW_DIR.glob('*.csv'))
csv_files

[WindowsPath('C:/Users/Ahmed/Documents/GazaSkyGeeksOlmpics/project/data/raw/assessments.csv'),
 WindowsPath('C:/Users/Ahmed/Documents/GazaSkyGeeksOlmpics/project/data/raw/courses.csv'),
 WindowsPath('C:/Users/Ahmed/Documents/GazaSkyGeeksOlmpics/project/data/raw/studentAssessment.csv'),
 WindowsPath('C:/Users/Ahmed/Documents/GazaSkyGeeksOlmpics/project/data/raw/studentInfo.csv'),
 WindowsPath('C:/Users/Ahmed/Documents/GazaSkyGeeksOlmpics/project/data/raw/studentRegistration.csv'),
 WindowsPath('C:/Users/Ahmed/Documents/GazaSkyGeeksOlmpics/project/data/raw/studentVle.csv'),
 WindowsPath('C:/Users/Ahmed/Documents/GazaSkyGeeksOlmpics/project/data/raw/vle.csv')]

## 1. Inspect files, columns, and row counts

In [2]:
overview = []

for path in csv_files:
    sample = pd.read_csv(path, nrows=5)
    rows = sum(1 for _ in open(path, 'rb')) - 1
    overview.append({
        'file': path.name,
        'rows': rows,
        'columns_count': len(sample.columns),
        'columns': ', '.join(sample.columns)
    })

pd.DataFrame(overview).sort_values('file')

,file,rows,columns_count,columns
0,assessments.csv,206,6,"code_module, code_presentation, id_assessment,..."
1,courses.csv,22,3,"code_module, code_presentation, module_present..."
2,studentAssessment.csv,173912,5,"id_assessment, id_student, date_submitted, is_..."
3,studentInfo.csv,32593,12,"code_module, code_presentation, id_student, ge..."
4,studentRegistration.csv,32593,5,"code_module, code_presentation, id_student, da..."
5,studentVle.csv,10655280,6,"code_module, code_presentation, id_student, id..."
6,vle.csv,6364,6,"id_site, code_module, code_presentation, activ..."


In [15]:
overview[1]

{'file': 'courses.csv',
 'rows': 22,
 'columns_count': 3,
 'columns': 'code_module, code_presentation, module_presentation_length'}

In [3]:
for path in csv_files:
    print(f'\n{path.name}')
    display(pd.read_csv(path, nrows=5))


assessments.csv


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19,10
1,AAA,2013J,1753,TMA,54,20
2,AAA,2013J,1754,TMA,117,20
3,AAA,2013J,1755,TMA,166,20
4,AAA,2013J,1756,TMA,215,30



courses.csv


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240



studentAssessment.csv


...

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159,NaN
1,AAA,2013J,28400,-53,NaN
2,AAA,2013J,30268,-92,12.0
3,AAA,2013J,31604,-52,NaN
4,AAA,2013J,32885,-176,NaN



studentVle.csv


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1



vle.csv


,id_site,code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,NaN,NaN
1,546712,AAA,2013J,oucontent,NaN,NaN
2,546998,AAA,2013J,resource,NaN,NaN
3,546888,AAA,2013J,url,NaN,NaN
4,547035,AAA,2013J,resource,NaN,NaN


## 2. Load tables

`studentVle.csv` is large, so it is read only when needed for aggregation.

In [4]:
courses = pd.read_csv(RAW_DIR / 'courses.csv')
assessments = pd.read_csv(RAW_DIR / 'assessments.csv')
student_info = pd.read_csv(RAW_DIR / 'studentInfo.csv')
student_registration = pd.read_csv(RAW_DIR / 'studentRegistration.csv')
student_assessment = pd.read_csv(RAW_DIR / 'studentAssessment.csv')
vle = pd.read_csv(RAW_DIR / 'vle.csv')

tables = {
    'courses': courses,
    'assessments': assessments,
    'student_info': student_info,
    'student_registration': student_registration,
    'student_assessment': student_assessment,
    'vle': vle,
}

pd.DataFrame({name: df.shape for name, df in tables.items()}, index=['rows', 'columns']).T

,rows,columns
courses,22,3
assessments,206,6
student_info,32593,12
student_registration,32593,5
student_assessment,173912,5
vle,6364,6


## 3. Missing values and categorical values

In [5]:
missing_summary = []

for name, df in tables.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    for column, count in missing.items():
        missing_summary.append({
            'table': name,
            'column': column,
            'missing_count': int(count),
            'missing_pct': round(count / len(df) * 100, 2)
        })

pd.DataFrame(missing_summary).sort_values(['table', 'column'])

,table,column,missing_count,missing_pct
0,assessments,date,11,5.34
4,student_assessment,score,173,0.10
1,student_info,imd_band,1111,3.41
2,student_registration,date_registration,45,0.14
3,student_registration,date_unregistration,22521,69.10
5,vle,week_from,5243,82.39
6,vle,week_to,5243,82.39


In [6]:
categorical_columns = {
    'student_info': ['gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability', 'final_result'],
    'assessments': ['assessment_type'],
    'vle': ['activity_type'],
}

for table_name, columns in categorical_columns.items():
    df = tables[table_name]
    print(f'\n{table_name}')
    for column in columns:
        print(f'\n{column}')
        display(df[column].value_counts(dropna=False).to_frame('count'))


student_info

gender


,count
gender,
M,17875
F,14718



region


,count
region,
Scotland,3446
East Anglian Region,3340
London Region,3216
South Region,3092
North Western Region,2906
West Midlands Region,2582
South West Region,2436
East Midlands Region,2365
South East Region,2111



highest_education


...

,count
final_result,
Pass,12361
Withdrawn,10156
Fail,7052
Distinction,3024



assessments

assessment_type


,count
assessment_type,
TMA,106
CMA,76
Exam,24



vle

activity_type


,count
activity_type,
resource,2660
subpage,1055
oucontent,996
url,886
forumng,194
quiz,127
page,102
oucollaborate,82
questionnaire,61


## 4. Relationship checks

Important keys:

- Course/presentation key: `code_module`, `code_presentation`
- Student course attempt key: `id_student`, `code_module`, `code_presentation`
- Assessment key: `id_assessment`
- VLE site key: `id_site`, `code_module`, `code_presentation`

In [7]:
student_key = ['id_student', 'code_module', 'code_presentation']
course_key = ['code_module', 'code_presentation']
site_key = ['id_site', 'code_module', 'code_presentation']

checks = {
    'student_info rows': len(student_info),
    'student_info unique student attempts': student_info[student_key].drop_duplicates().shape[0],
    'student_registration unique student attempts': student_registration[student_key].drop_duplicates().shape[0],
    'courses rows': len(courses),
    'courses unique course presentations': courses[course_key].drop_duplicates().shape[0],
    'assessments unique assessment ids': assessments['id_assessment'].nunique(),
    'vle unique site keys': vle[site_key].drop_duplicates().shape[0],
}

pd.Series(checks).to_frame('value')

,value
student_info rows,32593
student_info unique student attempts,32593
student_registration unique student attempts,32593
courses rows,22
courses unique course presentations,22
assessments unique assessment ids,206
vle unique site keys,6364


## 5. Build assessment features

`studentAssessment` does not contain module/presentation columns, so first join it to `assessments` by `id_assessment`. Then aggregate to the student course attempt grain.

In [8]:
assessment_events = student_assessment.merge(
    assessments,
    on='id_assessment',
    how='left',
    validate='many_to_one'
)

assessment_events['weighted_score_component'] = assessment_events['score'] * assessment_events['weight']
assessment_events['late_submission'] = assessment_events['date_submitted'] > assessment_events['date']

assessment_features = (
    assessment_events
    .groupby(student_key, as_index=False)
    .agg(
        assessment_submissions=('id_assessment', 'count'),
        assessment_score_mean=('score', 'mean'),
        assessment_score_min=('score', 'min'),
        assessment_score_max=('score', 'max'),
        assessment_weight_sum=('weight', 'sum'),
        assessment_weighted_score_sum=('weighted_score_component', 'sum'),
        late_submissions=('late_submission', 'sum'),
        banked_assessments=('is_banked', 'sum'),
    )
)

assessment_features['assessment_weighted_score_mean'] = (
    assessment_features['assessment_weighted_score_sum'] / assessment_features['assessment_weight_sum']
)

assessment_features.head()

,id_student,code_module,code_presentation,assessment_submissions,assessment_score_mean,assessment_score_min,assessment_score_max,assessment_weight_sum,assessment_weighted_score_sum,late_submissions,banked_assessments,assessment_weighted_score_mean
0,6516,AAA,2014J,5,61.800000,48.0,77.0,100.0,6350.0,0,0,63.50
1,8462,DDD,2013J,3,87.666667,83.0,93.0,40.0,3490.0,1,0,87.25
2,8462,DDD,2014J,4,86.500000,83.0,93.0,50.0,4300.0,0,4,86.00
3,11391,AAA,2013J,5,82.000000,78.0,85.0,100.0,8240.0,0,0,82.40
4,23629,BBB,2013B,4,82.500000,63.0,100.0,25.0,1669.0,3,0,66.76


In [9]:
assessment_type_features = (
    assessment_events
    .pivot_table(
        index=student_key,
        columns='assessment_type',
        values='score',
        aggfunc=['count', 'mean'],
        fill_value=0
    )
)

assessment_type_features.columns = [
    f'assessment_{metric.lower()}_{assessment_type.lower()}'
    for metric, assessment_type in assessment_type_features.columns
]

assessment_type_features = assessment_type_features.reset_index()
assessment_type_features.head()

,id_student,code_module,code_presentation,assessment_count_cma,assessment_count_exam,assessment_count_tma,assessment_mean_cma,assessment_mean_exam,assessment_mean_tma
0,6516,AAA,2014J,0,0,5,0.0,0.0,61.800000
1,8462,DDD,2013J,0,0,3,0.0,0.0,87.666667
2,8462,DDD,2014J,0,0,4,0.0,0.0,86.500000
3,11391,AAA,2013J,0,0,5,0.0,0.0,82.000000
4,23629,BBB,2013B,2,0,2,100.0,0.0,65.000000


## 6. Build VLE click features

`studentVle` is the largest table. Join it to `vle` so each click row has an `activity_type`, then aggregate to the student course attempt grain.

In [10]:
student_vle = pd.read_csv(RAW_DIR / 'studentVle.csv')

vle_events = student_vle.merge(
    vle,
    on=site_key,
    how='left',
    validate='many_to_one'
)

vle_features = (
    vle_events
    .groupby(student_key, as_index=False)
    .agg(
        vle_rows=('id_site', 'count'),
        total_clicks=('sum_click', 'sum'),
        active_days=('date', 'nunique'),
        first_vle_day=('date', 'min'),
        last_vle_day=('date', 'max'),
        unique_sites=('id_site', 'nunique'),
    )
)

vle_features['clicks_per_active_day'] = vle_features['total_clicks'] / vle_features['active_days']
vle_features.head()

,id_student,code_module,code_presentation,vle_rows,total_clicks,active_days,first_vle_day,last_vle_day,unique_sites,clicks_per_active_day
0,6516,AAA,2014J,662,2791,159,-23,269,84,17.553459
1,8462,DDD,2013J,300,646,56,-6,118,125,11.535714
2,8462,DDD,2014J,4,10,1,10,10,3,10.000000
3,11391,AAA,2013J,196,934,40,-5,253,55,23.350000
4,23629,BBB,2013B,59,161,16,-6,87,11,10.062500


In [11]:
vle_activity_features = (
    vle_events
    .pivot_table(
        index=student_key,
        columns='activity_type',
        values='sum_click',
        aggfunc='sum',
        fill_value=0
    )
)

vle_activity_features.columns = [f'clicks_{column}' for column in vle_activity_features.columns]
vle_activity_features = vle_activity_features.reset_index()
vle_activity_features.head()

,id_student,code_module,code_presentation,clicks_dataplus,clicks_dualpane,clicks_externalquiz,clicks_folder,clicks_forumng,clicks_glossary,clicks_homepage,...,clicks_ouelluminate,clicks_ouwiki,clicks_page,clicks_questionnaire,clicks_quiz,clicks_repeatactivity,clicks_resource,clicks_sharedsubpage,clicks_subpage,clicks_url
0,6516,AAA,2014J,21,0,0,0,451,0,497,...,0,0,0,0,0,0,31,0,143,143
1,8462,DDD,2013J,0,0,12,0,36,0,184,...,0,18,0,0,0,0,70,0,227,23
2,8462,DDD,2014J,0,0,0,0,2,0,7,...,0,0,0,0,0,0,0,0,0,0
3,11391,AAA,2013J,0,0,0,0,193,0,138,...,0,0,0,0,0,0,13,0,32,5
4,23629,BBB,2013B,0,0,0,0,87,0,36,...,0,0,0,0,31,0,2,0,5,0


## 7. Create the flat analysis table

Start from `studentInfo` because it contains the target variable `final_result` and already has one row per student course attempt. Add registration, course, assessment, and VLE features.

In [12]:
flat = (
    student_info
    .merge(student_registration, on=student_key, how='left', validate='one_to_one')
    .merge(courses, on=course_key, how='left', validate='many_to_one')
    .merge(assessment_features, on=student_key, how='left', validate='one_to_one')
    .merge(assessment_type_features, on=student_key, how='left', validate='one_to_one')
    .merge(vle_features, on=student_key, how='left', validate='one_to_one')
    .merge(vle_activity_features, on=student_key, how='left', validate='one_to_one')
)

feature_columns_to_fill = [
    column for column in flat.columns
    if column.startswith(('assessment_', 'vle_', 'clicks_'))
    or column in ['total_clicks', 'active_days', 'unique_sites', 'late_submissions', 'banked_assessments']
]

flat[feature_columns_to_fill] = flat[feature_columns_to_fill].fillna(0)

flat.shape, flat.head()

((32593, 57),
   code_module code_presentation  id_student gender                region  \
 0         AAA             2013J       11391      M   East Anglian Region   
 1         AAA             2013J       28400      F              Scotland   
 2         AAA             2013J       30268      F  North Western Region   
 3         AAA             2013J       31604      F     South East Region   
 4         AAA             2013J       32885      F  West Midlands Region   
 
        highest_education imd_band age_band  num_of_prev_attempts  \
 0       HE Qualification  90-100%     55<=                     0   
 1       HE Qualification   20-30%    35-55                     0   
 2  A Level or Equivalent   30-40%    35-55                     0   
 3  A Level or Equivalent   50-60%    35-55                     0   
 4     Lower Than A Level   50-60%     0-35                     0   
 
    studied_credits  ... clicks_ouelluminate clicks_ouwiki  clicks_page  \
 0              240  ...       

In [13]:
flat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 57 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   code_module                     32593 non-null  object 
 1   code_presentation               32593 non-null  object 
 2   id_student                      32593 non-null  int64  
 3   gender                          32593 non-null  object 
 4   region                          32593 non-null  object 
 5   highest_education               32593 non-null  object 
 6   imd_band                        31482 non-null  object 
 7   age_band                        32593 non-null  object 
 8   num_of_prev_attempts            32593 non-null  int64  
 9   studied_credits                 32593 non-null  int64  
 10  disability                      32593 non-null  object 
 11  final_result                    32593 non-null  object 
 12  date_registration               

## 8. Save later when ready

Uncomment the last line when the flat file design is final.

In [ ]:
# PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# flat.to_csv(PROCESSED_DIR / 'student_course_flat.csv', index=False)